# OpenPlaque — Left-Coronary Backbone Branch Discovery v1.1
Technical correction of the v1.0 vesselness-coordinate mismatch plus a guard for zero-length failed branch hypotheses. Scientific branch-discovery gates are unchanged. Run with **Runtime → Run all**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, shutil, sys

DRIVE_ROOT = Path('/content/drive/MyDrive/OpenPlaque')
OUTPUT = DRIVE_ROOT / 'Left_Coronary_Backbone_Branch_Discovery_v1'
REUSE_EXISTING_OUTPUT = False

if OUTPUT.exists() and not REUSE_EXISTING_OUTPUT:
    shutil.rmtree(OUTPUT)
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT / 'notebook_started.json').write_text(json.dumps({'status':'STARTED','notebook':'v1.1-short-path-fix'}, indent=2))
print('Output:', OUTPUT)

In [ ]:
BASELINE = '0593b453959f5a353d644267fbeef24b514ef4d7'
BRANCH = 'left-coronary-backbone-branch-discovery-from-main'
PINNED_SCIENCE_COMMIT = '7ab4357dbc012d58f1377362e9bbbdd57fb6eda2'
repo = '/content/OpenPlaque'
if os.path.exists(repo):
    shutil.rmtree(repo)
!git clone --depth 30 --branch "$BRANCH" https://github.com/pazzani/OpenPlaque.git "$repo"
!git -C "$repo" checkout --detach "$PINNED_SCIENCE_COMMIT"
HEAD = get_ipython().getoutput(f'git -C {repo} rev-parse HEAD')[0].strip()
MB = get_ipython().getoutput(f'git -C {repo} merge-base HEAD {BASELINE}')[0].strip()
print('Checked out:', HEAD)
print('Merge base:', MB)
assert HEAD == PINNED_SCIENCE_COMMIT
assert MB == BASELINE
%pip install -q /content/OpenPlaque
for name in list(sys.modules):
    if name == 'openplaque' or name.startswith('openplaque.'):
        del sys.modules[name]
os.chdir(repo)

In [ ]:
import numpy as np, pytest
from openplaque.left_coronary_backbone_branch_discovery_v1_1 import synthetic_local_vesselness_self_test, synthetic_short_path_self_test
print('Synthetic local-vesselness self-test:', synthetic_local_vesselness_self_test())
print('Synthetic short-path self-test:', synthetic_short_path_self_test())
rc = pytest.main(['-q', 'tests/test_left_coronary_backbone_branch_discovery_v1.py', 'tests/test_left_coronary_backbone_branch_discovery_v1_1.py'])
assert rc == 0, f'pytest failed with code {rc}'

In [ ]:
src_path = DRIVE_ROOT / 'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy'
old_ves_path = DRIVE_ROOT / 'Cache/Secondary_3D_Vesselness_Topology_v1/vesselness.npy'
src = np.load(src_path, mmap_mode='r')
old_ves = np.load(old_ves_path, mmap_mode='r')
print('Full Series-7 source shape:', src.shape)
print('Legacy cropped vesselness shape:', old_ves.shape)
assert src.shape != old_ves.shape, 'This v1.1 notebook is specifically for the known cropped-vesselness cache case.'
preflight = {
    'status':'COMPLETE',
    'science_commit': PINNED_SCIENCE_COMMIT,
    'source_shape': list(src.shape),
    'legacy_cached_vesselness_shape': list(old_ves.shape),
    'fix':'recompute local source-space vesselness around backbone; record one-point failed beams without numpy.gradient'
}
(OUTPUT / 'preflight_complete.json').write_text(json.dumps(preflight, indent=2))
del src, old_ves
print(json.dumps(preflight, indent=2))

In [ ]:
from openplaque.left_coronary_backbone_branch_discovery_v1_1 import run
result = run(drive_root=str(DRIVE_ROOT), output_dir=str(OUTPUT))
print(json.dumps(result['summary'], indent=2, default=str))
print('Report:', result['report'])
print('ZIP:', result['zip'])